In [2]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from notebookutils import mssparkutils

SILVER = "user_silver"
GOLD = "dim_user"

df_user = spark.table(SILVER)

df_new = (
    df_user.select(
        "user_id",
        "name",
        "review_count",
        "average_stars",
        "fans",
        "_ingest_ts"
    )
    .withColumn("is_current", F.lit(True))
    .withColumn("effective_from", F.current_timestamp())
    .withColumn("effective_to", F.lit(None).cast("timestamp"))
)

if not spark.catalog.tableExists(GOLD):
    (
    df_new.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD)
    )
    
    print(f"[INIT] {GOLD} created.")

else:
    delta = DeltaTable.forName(spark, GOLD)


    sccd_cols = ["name", "review_count", "average_stars", "fans"]
    
    change_cond = " OR ".join([f"current.{c}<> new.{c}" for c in sccd_cols])

    df_changed  = (
        spark.table(GOLD).filter(F.col("is_current") == True).alias("current")
        .join(df_new.alias("new"), "user_id")
        .filter(change_cond)
        .select("current.user_id")
    )

    (
        delta.alias("t")
        .merge(
            df_changed.alias("s"),
            "t.user_id = s.user_id AND t.is_current = true"
        )
        .whenMatchedUpdate(set={
            "is_current": "false",
            "effective_to": "current_timestamp()"
        })
        .execute()
    )

    df_existing_ids =  spark.table(GOLD).filter(F.col("is_current") == True).select("user_id")

    df_to_insert = (
        df_new.join(df_existing_ids, "user_id", "left_anti")
        .union(
            df_new.join(df_changed,  "user_id")
        )
    )

    (
        df_to_insert.write
        .format("delta")
        .mode("append")
        .saveAsTable(GOLD)
    )

    print(f"[SCD2] {GOLD} updated.  Changed: {df_changed.count()}, New: {df_new.join(df_existing_ids, 'user_id', 'left_anti').count()}")

mssparkutils.notebook.exit("SUCCESS")

StatementMeta(, 0fd31472-95b4-4caa-8e5c-d7ef42139155, 4, Finished, Available, Finished, False)

[INIT] dim_user created.
ExitValue: SUCCESS

In [3]:
spark.table("dim_business").groupBy("is_current").count().show()
spark.table("dim_user").groupBy("is_current").count().show()

StatementMeta(, 0fd31472-95b4-4caa-8e5c-d7ef42139155, 5, Finished, Available, Finished, False)

+----------+------+
|is_current| count|
+----------+------+
|      true|150346|
+----------+------+

+----------+-------+
|is_current|  count|
+----------+-------+
|      true|1987897|
+----------+-------+

